# Camera Hexapod Current Analysis

The purpose of this notebook is to study the camera hexapod forces (as measured by the strut currents) versus the commanded movements. LSSTCam has additional vacuum insulated pipes (VIP) that could cause additional torques with respect to ComCam ones.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import sys, time, os
import numpy as np
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

from astropy.time import Time
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

from bokeh.plotting import figure, show
from bokeh.models import HoverTool, TapTool, BoxZoomTool, ResetTool, ColumnDataSource
from bokeh.io import output_notebook, export_png

from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData

In [ ]:
plot_path = Path("./plots")
plot_path.mkdir(exist_ok=True, parents=True)

efd_client = makeEfdClient()
HEXAPOD_AXES = ["X", "Y", "Z", "U", "V", "W"]
N_AXES = len(HEXAPOD_AXES)
N_STRUTS = 6

## Helper Functions

In [ ]:
def units(ax: str) -> str:
    """Returns the camera hexapod units depending on the input string"""
    return "um" if ax.strip().upper() in "XYZ" else "º"

In [ ]:
def hide_axis(ax):
    """Hide axis labels, ticks, and spines for clean layout."""
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_frame_on(False)

In [ ]:
def compute_current_integrals(df):
    """Calculate the area (absolute) above and below zero"""
    df_cleaned = df.dropna()
    time_diffs = (df_cleaned.index[1:] - df_cleaned.index[:-1]).total_seconds()
    n_currents = N_STRUTS
    current_integrals = np.empty(n_currents)
    for idx in range(n_currents):
        reference_value = df_cleaned[f"motorCurrent{idx}"].iloc[0] 
        ##correct by value at the beginning of interval
        mean_y_values = ((df_cleaned[f"motorCurrent{idx}"].iloc[:-1].values 
                         + df_cleaned[f"motorCurrent{idx}"].iloc[1:].values) / 2) - reference_value
        current_integrals[idx] = np.sum(np.abs(mean_y_values)*time_diffs)
    return current_integrals


In [ ]:
def compute_delta_positions(df):
    """Calculate the difference between beginning and end positions and angles for hexapod"""
    df_cleaned = df.dropna()
    delta_positions = np.zeros(N_AXES)
    for i,ax in enumerate(HEXAPOD_AXES):
        delta_positions[i] = df_cleaned[ax].iloc[-1] - df_cleaned[ax].iloc[0]
    return delta_positions

In [ ]:
def plot_hexapod(df, title, smooth_currents=False):
    """
    Plot hexapod data as a time series.
    This will show the positions and the currents.
    """
    nrows = 3
    ncols = 2

    current_integrals = compute_current_integrals(df)

    # Create figure and the grid for the plots
    fig = plt.figure(num=title, figsize=(10, 6))
    gs_outer = GridSpec(nrows=1, ncols=2, figure=fig)

    ax_title_left = fig.add_subplot(gs_outer[0])
    ax_title_left.set_title("CamHex Positions")
    hide_axis(ax_title_left)
    
    ax_title_right = fig.add_subplot(gs_outer[1])
    ax_title_right.set_title("Motor Currents in Struts")
    hide_axis(ax_title_right)
    
    gs_left = GridSpecFromSubplotSpec(
        nrows=nrows, ncols=ncols, subplot_spec=gs_outer[0], hspace=0, wspace=0.05
    )
    gs_right = GridSpecFromSubplotSpec(
        nrows=nrows, ncols=ncols, subplot_spec=gs_outer[1], hspace=0, wspace=0.05
    )

    # Placeholders for each plot
    axes_left = [[None for _ in range(2)] for _ in range(3)]
    axes_right = [[None for _ in range(2)] for _ in range(3)]

    # Determine the time range
    time_range = (df.index[-1] - df.index[0]).total_seconds() / 60  # Convert to minutes

    # Populate the plots
    for j in range(ncols):
        for i in range(nrows):
        
            idx = i + nrows * j
            hax = HEXAPOD_AXES[idx]

            # Left plots -- Positions --
            ax = fig.add_subplot(
                gs_left[i, j],
                sharex=axes_left[0][0] if axes_left[0][0] else None,
            )
            ax.plot(df[hax], color="royalblue")
            ax.grid(":", alpha=0.25)
            ax.set_ylabel(f"{hax} [{units(hax)}]")
            ax.tick_params(axis='both', labelsize=8)
            axes_left[i][j] = ax

            # Move ylabels and yticklabels to the right for the second column
            if j == 1:
                ax.yaxis.set_label_position("right")
                ax.yaxis.tick_right()

            # Right plots -- Currents --
            ax = fig.add_subplot(
                gs_right[i, j],
                sharex=axes_right[0][0] if axes_right[0][0] else None,
            )
            ax.plot(df[f"motorCurrent{idx}"], color="darkgreen")
            plt.text(0.35, 0.1, f"abs. area = {current_integrals[idx]:.3f}", 
                     fontsize=9, color='green', ha='center', va='center', 
                     transform=plt.gca().transAxes, 
                     bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))  
            ax.grid(":", alpha=0.25)
            ax.set_ylim(-5.9, 5.9)
            ax.set_ylabel(f"Strut #{idx} [A]")
            ax.tick_params(axis='both', labelsize=8)
            axes_right[i][j] = ax

            # Move ylabels and yticklabels to the right for the second column
            if j == 1:
                ax.yaxis.set_label_position("right")
                ax.yaxis.tick_right()

        # Apply different x-tick formatting based on timespan
        if time_range > 60:
            axes_left[-1][j].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
            axes_right[-1][j].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        elif time_range < 5:
            axes_left[-1][j].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
            axes_right[-1][j].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    
        axes_left[-1][j].set_xlabel("Time [UTC]")
        axes_right[-1][j].set_xlabel("Time [UTC]")
    
    fig.suptitle(f"{title}\nFrom {Time(df.index[0]).isot} to {Time(df.index[-1]).isot}")
    fig.autofmt_xdate()
    fig.tight_layout()
    
    return fig

In [ ]:
def plot_current_vs_movements(avg_charge,deltas,t_movements,title):
    """
    Plot hexapod average current across struts vs 
    the amount of movement in position for the 
    axis with maximum displacement. 
    Plot histogram of 'efficiency' of movement (Amperes per micron of movement)
    avg_charge = area under the curve of the current vs time for each movements
    deltas = period of time for each movement
    t_movements = start and end of each movement
    title = for the plots
    """
    ## Compute some derived quantities
    # maximum displacement among three position axes per movement
    max_delta_position = [np.max(abs(arr[:3])) for arr in deltas]
    # time duration in seconds for movement
    time_diff = [(arr[1]-arr[0]).sec for arr in t_movements]
    # start and end of each movement
    t0 = [str(arr[0]) for arr in t_movements]
    t1 = [str(arr[1]) for arr in t_movements]    
    # total current for each movement
    avg_current = np.array(avg_charge)/np.array(time_diff)
    # 'efficiency' current per micron
    efficiency = avg_current/max_delta_position
    
    deltas_column = [[deltas[i][j] for i in range(len(deltas))] for j in range(len(deltas[0]))]
    source = ColumnDataSource(data={'x':avg_current,
                                    'y':max_delta_position,
                                    'delta_x':deltas_column[0], 
                                    'delta_y':deltas_column[1], 
                                    'delta_z':deltas_column[2],
                                    't0':t0,
                                    't1':t1,
                                    'delta_time':time_diff})
    hover_opts = dict(tooltips=[("(Delta_x,Delta_y,Delta_z,,t0,t1,Delta_time)", 
                                 "(@delta_x,@delta_y,@delta_z,@t0,@t1,@delta_time)")],
                      show_arrow=False, line_policy="next")
    p = figure(title=f"Average current for all movements {title}", 
               x_axis_label="Average current [A]",
               y_axis_label="Max. movement [um]",
               tools=[HoverTool(**hover_opts), TapTool(), BoxZoomTool(), ResetTool()],
               ) 

    p.scatter(x='x',y='y', source=source)
    
    output_notebook()
    show(p)
    ## exporting png requires some additional dependencies: 
    ## https://docs.bokeh.org/en/latest/docs/user_guide/output/export.html#ug-output-export
    #export_png(p, filename=f"{plot_path}/SITCOM-1946_current_vs_movement_scatter.png")
    

    histmax, edges = np.histogram(efficiency,bins=20)
    bin_centers = (edges[:-1] + edges[1:]) / 2
    p2 = figure(title=f"Efficiency (current per micron {title})",
               x_axis_label="Efficiency [A/um]",
               tools=[BoxZoomTool(), ResetTool()],
               )
    source_hist = ColumnDataSource(data={'bin_centers':bin_centers, 'top':histmax, 
                                    'left': edges[:-1], 'right': edges[1:],})
    p2.quad(source=source_hist, bottom=0, line_color="blue")

    show(p2)
    #export_png(p, filename=f"{plot_path}/SITCOM-1946_current_vs_movement_eff_histo.png")



In [ ]:
def get_movements(t_start_period, t_end_period, use_command = True):
    """
    Retrieve start and end of all hexapod movements, 
    within a period given by t_start_period, t_end_period
    """
    # two options are provided to define the beginning of the movement
    # the first (use_command = True) determines the start of the movement when command_move is issued
    # the second (use_command = False) determined the start of the movement when controller state is PointToPoint
    if use_command:
        df_state = getEfdData(client=efd_client, 
                      topic="lsst.sal.MTHexapod.command_move",
                      begin=t_start_period,
                      end=t_end_period,
        )
        df_state_selected = df_state
    else:
        df_state = getEfdData(client=efd_client, 
                      topic="lsst.sal.MTHexapod.logevent_controllerState", 
                      begin=t_start_period,
                      end=t_end_period,
        )
        #as condition for start, retrieve the first time that the controller is ENABLED
        # and substate is MovingPointToPoint (can be expanded to Tracking/Slewing)
        #(see https://ts-xml.lsst.io/sal_interfaces/MTHexapod.html#enumerations)
        condition_movement = (df_state['controllerState'] == 2) & (df_state['enabledSubstate'] == 1)
        df_state_selected = df_state[condition_movement]  

    n_state_events_mov = len(df_state_selected)
    t_movements = [(None,None)] * n_state_events_mov

    for i in range(n_state_events_mov):
        entry = df_state_selected.iloc[i] if not df_state_selected.empty else None
        if entry is None:
            t_movements.pop(i)
            continue
        t_start = Time(entry.name, scale="utc")
        df_inposition =  getEfdData(client=efd_client, 
                     topic="lsst.sal.MTHexapod.logevent_inPosition", 
                     begin=t_start,
                     end=t_end_period,
        )
        
        #whenever the hexapod logs as "in position" immediately after t_start, register that time
        condition_inposition = (df_inposition["inPosition"] == True) 
        entry = df_inposition[condition_inposition].iloc[0] if not df_inposition[condition_inposition].empty else None
        if entry is None:
            t_movements.pop(i)
            continue
        t_end = Time(entry.name, scale="utc")
        delta = - df_state_selected["x"]
        #print(delta)
        t_movements[i] = (t_start,t_end,delta)
    
    return t_movements


In [ ]:
def query_data(start, end):
    df_position = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTHexapod.application",
        columns=[f"position{i}" for i in range(N_AXES)] + ["salIndex"],
        begin=start,
        end=end,
    )
    df_position = df_position[df_position["salIndex"] == 1]
    df_position = df_position.drop(columns="salIndex")
    df_position = df_position.rename(
        columns={f"position{i}": f"{ax}" for i, ax in enumerate(HEXAPOD_AXES)}
    )
    
    df_currents = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTHexapod.electrical",
        columns=[f"motorCurrent{i}" for i in range(N_STRUTS)] + ["salIndex"],
        begin=start,
        end=end,
    )
    df_currents = df_currents[df_currents["salIndex"] == 1]
    df_currents = df_currents.drop(columns="salIndex")
    df = pd.merge_asof(
        left=df_position, right=df_currents, left_index=True, right_index=True
    )
    del df_currents, df_position

    return df

## Camera Hexapod movements vs torques 

Retrieve all movements given a set period. These movements are defined as the periods between when command_move is sent, and the inPosition event is set. This cell then plots the measured current (torque) vs maximum displacement of each movement, and the efficiency as the current passing measured in the system per micron.

In [ ]:
t_start_period = Time("2025-02-27T22:07:00Z", scale="utc") #LSSTCam warm up
t_end_period = Time("2025-02-27T22:20:00Z", scale="utc") #LSSTCam warm up
t_movements = get_movements(t_start_period, t_end_period, use_command=True)
nb_movements = len(t_movements)
print(f"Found {nb_movements} movement(s)")
avg_charge = []
deltas = []
t_movements_selected = []
for i in range(nb_movements):
    df = query_data(t_movements[i][0],t_movements[i][1])
    delta_pos = compute_delta_positions(df)
    if np.any(abs(delta_pos[:3]) > 100): # in microns. Only large, commanded movements are plotted
        deltas.append(delta_pos)
        charge = compute_current_integrals(df)
        avg_charge.append(np.mean(charge))
        #total_charge.append(charge) #TBD version for one specific strut
        t_movements_selected.append([t_movements[i][0],t_movements[i][1]])
t_start_period.format = 'iso'
t_start_period.out_subfmt = 'date_hms'
t_start_period.precision = 0
t_end_period.format = 'iso'
t_end_period.out_subfmt = 'date_hms'
t_end_period.precision = 0
title = f"{t_start_period.value} - {t_end_period.value}"
plot_current_vs_movements(avg_charge,deltas,t_movements_selected,title)

Same thing with a set of ComCam measurements

Other possible periods, besides the one used below: 

#t_start_period = Time("2024-12-07T20:38:00Z", scale="utc") #ComCam warm up

#t_end_period = Time("2024-12-07T20:51:00Z", scale="utc") #ComCam warm up

#t_start_period = Time("2024-12-08T21:11:00Z", scale="utc") #ComCam warm up

#t_end_period = Time("2024-12-08T21:22:00Z", scale="utc") #ComCam warm up

#t_start_period = Time("2024-12-09T17:40:00Z", scale="utc") #ComCam warm up

#t_end_period = Time("2024-12-09T17:51:00Z", scale="utc") #ComCam warm up

#t_start_period = Time("2024-12-09T22:33:00Z", scale="utc") #ComCam warm up

#t_end_period = Time("2024-12-09T22:44:00Z", scale="utc") #ComCam warm up

#t_start_period = Time("2024-12-10T16:58:00Z", scale="utc") #ComCam warm up

#t_end_period = Time("2024-12-10T17:10:00Z", scale="utc") #ComCam warm up


In [ ]:
t_start_period = Time("2024-12-06T18:00:00Z", scale="utc") #ComCam warm up
t_end_period = Time("2024-12-06T18:20:00Z", scale="utc") #ComCam warm up
t_movements = get_movements(t_start_period, t_end_period, use_command=True)
nb_movements = len(t_movements)
print(f"Found {nb_movements} movement(s)")
avg_charge = []
deltas = []
t_movements_selected = []
minimum_delta_pos = 10 # in microns, use larger thresholds to select significant commanded movements
for i in range(nb_movements):
    df = query_data(t_movements[i][0],t_movements[i][1])
    delta_pos = compute_delta_positions(df)
    if np.any(abs(delta_pos[:3]) > minimum_delta_pos): 
        deltas.append(delta_pos)
        charge = compute_current_integrals(df)
        avg_charge.append(np.mean(charge)) # average charge across all struts. TBD version for one specific strut
        t_movements_selected.append([t_movements[i][0],t_movements[i][1]])
t_start_period.format = 'iso'
t_start_period.out_subfmt = 'date_hms'
t_start_period.precision = 0
t_end_period.format = 'iso'
t_end_period.out_subfmt = 'date_hms'
t_end_period.precision = 0
title = f"{t_start_period.value} - {t_end_period.value}"
if len(deltas) < 1:
    print("No movement above threshold")
else:
    plot_current_vs_movements(avg_charge,deltas,t_movements_selected,title)

## The following cells illustrate time series for movements and currents for different periods

Some of these tests are imported from SITCOM-1883 notebooks.

### Hexapod test during movement event

In [ ]:
delta_positions = np.zeros(N_AXES)
print(delta_positions)

In [ ]:
# identify time period in which to look for a movement event
t_start_period = Time("2025-02-26T17:00:00Z", scale="utc")
t_end_period = Time("2025-02-26T19:00:00Z", scale="utc")

# identify movements in time period
t_movements = get_movements(t_start_period, t_end_period)

# choose a particular movement in the period
selected_movement = 0
t_start = t_movements[selected_movement][0]
t_end = t_movements[selected_movement][1]

# fill dataframe with selected movement data
df = query_data(t_start, t_end)
fig = plot_hexapod(df, title="LSSTCam Testing")
plt.show()

### LSSTCam Hexapod testing with no rotations

In [ ]:
t_start = Time("2025-02-26T17:00:00Z", scale="utc")
t_end = Time("2025-02-26T19:00:00Z", scale="utc")

df = query_data(t_start, t_end)

fig = plot_hexapod(df, title="LSSTCam Testing 2025-02-26")
plt.show()

### LSSTCam Hexapod testing with maximal rotations


In [ ]:
t_start = Time("2025-02-27T13:00:00Z", scale="utc")
t_end = Time("2025-02-27T18:00:00Z", scale="utc")

df = query_data(t_start, t_end)

fig = plot_hexapod(df, title="LSSTCam Testing 2025-02-27")
plt.show()

### Typical ComCam observing night.

In [ ]:
t_start = Time("2024-12-12T05:00:00Z", scale="utc")
t_end = Time("2024-12-12T07:00:00Z", scale="utc")

df = query_data(t_start, t_end)

fig = plot_hexapod(df, title="ComCam observing 2024-12-11")
plt.show()

### Blowup of LSSTCam Hexapod testing with no rotations

In [ ]:
t_start = Time("2025-02-26T17:37:00Z", scale="utc")
t_end = Time("2025-02-26T17:39:00Z", scale="utc")

df = query_data(t_start, t_end)

fig = plot_hexapod(df, title="LSSTCam Testing 2025-02-26 Zoomed")
plt.show()

### Hexapod warm-up - LSSTCam

In [ ]:
t_start = Time("2025-02-27T22:09:10Z", scale="utc")
t_end = Time("2025-02-27T22:09:14Z", scale="utc")

df = query_data(t_start, t_end)

fig = plot_hexapod(df, title="LSSTCam Hexapod Warm-up 2025-02-27")
plt.show()

### Hexapod warm-up - ComCam

In [ ]:
t_start = Time("2024-12-06T18:06:07Z", scale="utc")
t_end = Time("2024-12-06T18:06:11Z", scale="utc")

df = query_data(t_start, t_end)

fig = plot_hexapod(df, title="ComCam Hexapod Zoom after Warm-up 2024-12-06")
plt.show()